# Combined CRF and Focal loss


In [ ]:
!pip install transformers seqeval evaluate accelerate -U
!pip install transformers seqeval evaluate accelerate pytorch-crf -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1cd8ed01643308b6c9d4860c1c21413eeda513184d56e74eaf518140e13fb4d4
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'
train_path = '/content/drive/MyDrive/datasetViMedNER/traindata/train.txt'
dev_path = '/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt'

unique_tags = []
with open(label_path, "r", encoding = "utf-8") as f :
    for line in f:
        line.strip()
        if line.strip():
          unique_tags.append(line.strip())

label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

train_sentences = load_conll_data(train_path)
dev_sentences = load_conll_data(dev_path)

In [ ]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from IPython.display import display

seqeval = evaluate.load("seqeval")

def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
import pickle
train_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/train_dataset.pkl'
dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'

def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset


train_dataset = load_dataset(train_dataset_filepath)
dev_dataset = load_dataset(dev_dataset_filepath)

CRF + Focal

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets, mask):
        logits = logits.view(-1, logits.size(-1))
        targets = targets.view(-1)
        mask = mask.view(-1)

        logits = logits[mask]
        targets = targets[mask]

        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class PhoBERT_Focal_CRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels, gamma=2.0, alpha=None):
        super().__init__()
        self.num_labels = num_labels
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)

        # 1. Khởi tạo CRF layer
        self.crf = CRF(num_labels, batch_first=True)

        # 2. Khởi tạo Focal Loss
        self.focal_loss = FocalLoss(gamma=gamma, alpha=alpha)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)

        # Truyền thẳng đầu ra (không qua Dropout) vào bộ phân loại
        sequence_output = outputs.last_hidden_state
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # A. TÍNH CRF LOSS (Sequence-level)
            crf_mask = attention_mask.bool()
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 10 # Gán tạm nhãn 0 cho padding/sub_token
            crf_loss = -self.crf(logits, safe_labels, mask=crf_mask, reduction='mean')

            # B. TÍNH FOCAL LOSS (Token-level)
            focal_mask = (labels != -100) # Chỉ tính loss trên các token thật
            focal_loss_val = self.focal_loss(logits, labels, focal_mask)

            # C. TỔNG HỢP LOSS
            loss = crf_loss + focal_loss_val
        # D. GIẢI MÃ (DECODE) BẰNG CRF ĐỂ ĐÁNH GIÁ
        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(logits, mask=crf_mask_decode)

        # Tạo fake_logits để tương thích hoàn toàn với hàm compute_metrics
        fake_logits = torch.zeros_like(logits)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

KeyboardInterrupt: 

In [ ]:
from torch.optim import AdamW
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
# 1. Khởi tạo mô hình kết hợp
model_combined = PhoBERT_Focal_CRF(model_checkpoint="vinai/phobert-base-v2", num_labels=len(unique_tags))

# 2. Khai báo Optimizer tách biệt Learning Rate
crf_params = list(model_combined.crf.parameters())
phobert_classifier_params = list(model_combined.phobert.parameters()) + list(model_combined.classifier.parameters())

optimizer_grouped_parameters = [
    {'params': phobert_classifier_params, 'lr': 3e-5}, # LR nhỏ cho PhoBERT
    {'params': crf_params, 'lr': 3e-3}                 # LR lớn cho CRF
]
optimizer_combined = AdamW(optimizer_grouped_parameters, weight_decay=0.01)

# 3. Cấu hình Training
training_args_combined = TrainingArguments(
    output_dir="./phobert_focal_crf_model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,               # Đặt số epoch lớn, Early Stopping sẽ lo phần dừng
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    max_grad_norm=1.0,
    report_to="none"
)

# 4. Khởi tạo Trainer
trainer_combined = Trainer(
    model=model_combined,
    args=training_args_combined,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_combined, None), # Dùng Optimizer đã tách LR
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Bạn có thể chạy thẳng lệnh này ở 1 Cell mới
trainer_combined.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,11.197738,0.641779,0.662282,0.651869,0.882242
2,16.357044,8.681042,0.650382,0.686174,0.667799,0.889524
3,16.357044,7.014665,0.730534,0.705235,0.717662,0.900385
4,6.203768,6.821453,0.675075,0.725638,0.699444,0.885838
5,6.203768,7.094429,0.669303,0.749799,0.707268,0.891108
6,3.459988,7.312468,0.704884,0.724564,0.714588,0.896663


TrainOutput(global_step=1716, training_loss=7.915480927153901, metrics={'train_runtime': 1343.0885, 'train_samples_per_second': 51.073, 'train_steps_per_second': 3.194, 'total_flos': 0.0, 'train_loss': 7.915480927153901, 'epoch': 6.0})

In [ ]:
import numpy as np
import os

print("🚀 ĐANG ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP DEV...")
# 1. Lấy dự đoán từ mô hình kết hợp
pred_results = trainer_combined.predict(dev_dataset)

# 2. In bảng phân tích độ đo chuyên sâu (Entity-level & Tag-level)
# Lưu ý: truyền vào dạng tuple (predictions, label_ids)
compute_eval_classify_metrics((pred_results.predictions, pred_results.label_ids))

# 3. Xuất kết quả ra file text để phân tích lỗi (Error Analysis)
output_dir = "/content/drive/MyDrive/datasetViMedNER/"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "focal_crf_dev_analysis.txt")

predictions = np.argmax(pred_results.predictions, axis=2)
labels = pred_results.label_ids

predicted_tags_per_sentence = []

# Lọc bỏ các token padding/sub-word (-100) và chuyển ID về dạng Text
for prediction, label in zip(predictions, labels):
    pred_sent = [id2label[p] for p, l in zip(prediction, label) if l != -100]
    predicted_tags_per_sentence.append(pred_sent)

print(f"\n🔍 Đang xuất file đối chiếu lỗi...")
with open(output_file, "w", encoding="utf-8") as f:
    # dev_sentences là list các câu gốc chứa tuple (word, true_tag)
    for idx, (original_sent, predicted_tags) in enumerate(zip(dev_sentences, predicted_tags_per_sentence)):
        if len(original_sent) == len(predicted_tags):
            for (word, true_tag), pred_tag in zip(original_sent, predicted_tags):
                # Ghi theo định dạng: Từ \t Nhãn thật \t Nhãn dự đoán
                f.write(f"{word}\t{true_tag}\t{pred_tag}\n")
            f.write("\n") # Dòng trống để ngăn cách các câu
        else:
            print(f"⚠️ Cảnh báo: Lệch số lượng từ ở câu {idx}, bỏ qua xuất câu này...")

print(f"✅ Đã xuất file đối chiếu thành công tại: {output_file}")

🚀 ĐANG ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP DEV...



📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.6052,0.6900,0.6448,271
1,bien_phap_dieu_tri,0.6081,0.6631,0.6344,653
2,nguyen_nhan_benh,0.5283,0.1081,0.1795,259
3,ten_benh,0.8051,0.8585,0.8310,1795
4,trieu_chung_benh,0.7204,0.5863,0.6465,747
5,OVERALL,0.7305,0.7052,0.7177,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7079,0.7601,0.7331,271
1,B-bien_phap_dieu_tri,0.6884,0.7443,0.7152,653
2,B-nguyen_nhan_benh,0.6735,0.1274,0.2143,259
3,B-ten_benh,0.8430,0.8975,0.8694,1795
4,B-trieu_chung_benh,0.7774,0.6265,0.6938,747
5,I-bien_phap_chan_doan,0.7133,0.6815,0.6971,942
6,I-bien_phap_dieu_tri,0.6416,0.5828,0.6108,1745
7,I-nguyen_nhan_benh,0.7440,0.1094,0.1908,850
8,I-ten_benh,0.8791,0.9106,0.8946,4585
9,I-trieu_chung_benh,0.7429,0.5355,0.6224,1522



🔍 Đang xuất file đối chiếu lỗi...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 63, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 293, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 1459, bỏ qua xuất câu này...
✅ Đã xuất file đối chiếu thành công tại: /content/drive/MyDrive/datasetViMedNER/focal_crf_dev_analysis.txt


In [ ]:
import os
import torch

# Đường dẫn lưu model trên Drive
save_dir = "/content/drive/MyDrive/phobert_focal_crf"
os.makedirs(save_dir, exist_ok=True)

print("💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...")
tokenizer.save_pretrained(save_dir)

# Lưu toàn bộ state_dict của mô hình (bao gồm cả PhoBERT, Classifier, CRF và Focal Loss)
torch.save(trainer_combined.model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
print(f"✅ Đã lưu mô hình thành công tại: {save_dir}")

💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...
✅ Đã lưu mô hình thành công tại: /content/drive/MyDrive/phobert_focal_crf
